<a href="https://www.kaggle.com/code/aleksandrmorozov123/ai-agents?scriptVersionId=251523820" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Implementation of reflection patterns on Groq models**

In [2]:
! pip install groq 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.2 MB/s eta 0:00:00


In [3]:
!pip install python-dotenv

In [4]:
import os
import re
import math
import json
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")

# Remember to load the environment variables
load_dotenv()

client = Groq(api_key=secret_value_0)

In [5]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

In [6]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Bucket Sort algorithm"
    }
)

In [7]:
bucketsort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": bucketsort_code
    }
)

In [8]:
display_markdown(bucketsort_code, raw=True)

Here's an implementation of the Bucket Sort algorithm in Python:
```
def bucket_sort(arr):
    # Create empty buckets
    buckets = [[] for _ in range(len(arr))]

    # Insert elements into their respective buckets
    for ele in arr:
        index = int(ele * len(buckets))  # mapping elements to bucket indices
        buckets[index].append(ele)

    # Sort individual buckets
    for i in range(len(buckets)):
        buckets[i] = sorted(buckets[i])

    # Concatenate the result
    result = []
    for bucket in buckets:
        result.extend(bucket)

    return result

# Example usage
arr = [0.897, 0.565, 0.656, 0.1234, 0.665, 0.3434]
sorted_arr = bucket_sort(arr)
print("Sorted array is:", sorted_arr)
```
This implementation assumes that the input array consists of floating-point numbers between 0 and 1. The `bucket_sort` function works as follows:

1. It creates an array of empty buckets, where each bucket will hold elements within a specific range.
2. It inserts each element from the input array into its corresponding bucket based on its value. The bucket index is calculated by multiplying the element value with the number of buckets.
3. It sorts each individual bucket using the built-in `sorted` function.
4. Finally, it concatenates the sorted buckets to produce the sorted output array.

Note that Bucket Sort is not a comparison-based sort and has a average time complexity of O(n + k), where n is the number of elements and k is the number of buckets.

In [9]:
# reflection part
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are an experienced phD computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": bucketsort_code
    }
)

In [11]:
# generate critique to the python code
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [12]:
display_markdown(critique, raw=True)

Here's a critique of the provided implementation of the Bucket Sort algorithm:

**Critique:**

1. **Magic numbers:** The implementation assumes that the input array consists of floating-point numbers between 0 and 1. This assumption should be explicitly mentioned in the function's documentation, and ideally, the function should be able to handle input arrays with any range of values.

2. **Bucket index calculation:** The calculation of the bucket index using `int(ele * len(buckets))` can lead to poor distribution of elements among buckets if the input array has a skewed distribution. A more robust approach would be to use a hash function that distributes the elements more uniformly across buckets.

3. **Choice of number of buckets:** The number of buckets is set to the length of the input array, which may not be the optimal choice. A more informed decision could be made based on the range of values in the input array.

4. **Use of built-in `sorted` function:** While this implementation uses the built-in `sorted` function to sort individual buckets, it's worth noting that Bucket Sort's average time complexity of O(n + k) is achieved when the buckets are sorted using an insertion sort or a similar linear-time sorting algorithm.

5. **Lack of error handling:** The implementation does not perform any error checking on the input array. It assumes that the input array is a list of numbers, which might not always be the case.

6. **Code organization and naming:** The implementation could benefit from more descriptive variable names and functions. For example, `bucket_sort` could be renamed to `bucket_sort_floating_point_numbers`, and the inner loop could be extracted into a separate function.

**Recommendations:**

1. **Input validation:** Add input validation to check that the input array is a list of numbers and handle any potential errors.

2. **Dynamic bucket size calculation:** Calculate the number of buckets dynamically based on the range of values in the input array.

3. **Robust bucket index calculation:** Implement a more robust hash function to distribute elements uniformly across buckets.

4. **Custom bucket sorting:** Implement a custom sorting algorithm (e.g., insertion sort) to sort individual buckets, which will help achieve the O(n + k) time complexity.

5. **Code organization and naming:** Improve code organization by using more descriptive variable names and extracting functions for better readability and maintainability.

Here's an updated implementation addressing these concerns:
```
def bucket_sort_floating_point_numbers(arr):
    """
    Sorts an array of floating-point numbers between 0 and 1 using Bucket Sort.

    Args:
        arr (list): Input array of floating-point numbers.

    Returns:
        list: Sorted array.
    """

    # Input validation
    if not isinstance(arr, list) or not all(isinstance(x, (int, float)) for x in arr):
        raise ValueError("Input must be a list of numbers")

    # Calculate the range of values
    min_val = min(arr)
    max_val = max(arr)

    # Calculate the number of buckets dynamically
    num_buckets = len(arr)
    buckets = [[] for _ in range(num_buckets)]

    # Insert elements into their respective buckets
    for ele in arr:
        # Robust bucket index calculation
        index = int((ele - min_val) / (max_val - min_val) * (num_buckets - 1))
        buckets[index].append(ele)

    # Sort individual buckets using insertion sort
    for i in range(len(buckets)):
        insertion_sort(buckets[i])

    # Concatenate the result
    result = []
    for bucket in buckets:
        result.extend(bucket)

    return result

def insertion_sort(arr):
    """
    Sorts an array using insertion sort.

    Args:
        arr (list): Input array.

    Returns:
        list: Sorted array.
    """
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr

# Example usage
arr = [0.897, 0.565, 0.656, 0.1234, 0.665, 0.3434]
sorted_arr = bucket_sort_floating_point_numbers(arr)
print("Sorted array is:", sorted_arr)
```

In [13]:
# add critique to the generation chat history
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [14]:
# generation step 2
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [15]:
display_markdown(essay, raw=True)

Here's a revised implementation that addresses the critique and recommendations:
```
def bucket_sort(arr):
    """
    Sorts an array of numbers using Bucket Sort.

    Args:
        arr (list): Input array of numbers.

    Returns:
        list: Sorted array.
    """

    # Input validation
    if not isinstance(arr, list) or not all(isinstance(x, (int, float)) for x in arr):
        raise ValueError("Input must be a list of numbers")

    # Calculate the range of values
    min_val = min(arr)
    max_val = max(arr)

    # Calculate the number of buckets dynamically
    num_buckets = int(len(arr) ** 0.5)
    buckets = [[] for _ in range(num_buckets)]

    # Insert elements into their respective buckets
    for ele in arr:
        # Robust bucket index calculation
        index = int((ele - min_val) / (max_val - min_val) * (num_buckets - 1))
        buckets[index].append(ele)

    # Sort individual buckets using insertion sort
    for i in range(len(buckets)):
        buckets[i] = insertion_sort(buckets[i])

    # Concatenate the result
    result = []
    for bucket in buckets:
        result.extend(bucket)

    return result

def insertion_sort(arr):
    """
    Sorts an array using insertion sort.

    Args:
        arr (list): Input array.

    Returns:
        list: Sorted array.
    """
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr

# Example usage
arr = [0.897, 0.565, 0.656, 0.1234, 0.665, 0.3434]
sorted_arr = bucket_sort(arr)
print("Sorted array is:", sorted_arr)
```
The revised implementation addresses the following concerns:

1. **Magic numbers:** The implementation no longer assumes that the input array consists of floating-point numbers between 0 and 1. Instead, it calculates the range of values dynamically.

2. **Bucket index calculation:** A more robust hash function is used to distribute elements uniformly across buckets.

3. **Choice of number of buckets:** The number of buckets is calculated dynamically based on the length of the input array.

4. **Use of built-in `sorted` function:** The implementation uses a custom insertion sort function to sort individual buckets, which helps achieve the O(n + k) time complexity.

5. **Lack of error handling:** Input validation is added to check that the input array is a list of numbers and handle any potential errors.

6. **Code organization and naming:** The implementation uses more descriptive variable names and function names, and the inner loop is extracted into a separate function for better readability and maintainability.

**Implement tool function for agent**

In [16]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Miami, Florida
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Miami":
		return json.dumps({"Fahrenheit": 75, "unit": unit})

	else:
		return json.dumps({"temperature": 130, "unit": unit})

In [17]:
get_current_weather(location="Miami", unit="Fahrenheit")

'{"Fahrenheit": 75, "unit": "Fahrenheit"}'

In [18]:
MODEL = "llama-3.3-70b-versatile"
GROQ_CLIENT = Groq(api_key=secret_value_0)

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [19]:
# ask the question
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Miami, in Fahrenheit?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}}
</tool_call>


In [20]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [21]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Miami', 'unit': 'fahrenheit'}}

In [22]:
result = get_current_weather(**parsed_output["arguments"])

In [23]:
result

'{"Fahrenheit": 75, "unit": "fahrenheit"}'

In [24]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [25]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Miami is 75°F.'

**Implementation of planning pattern**

In [26]:
!pip install agents

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.6/118.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 17.7 MB/s eta 0:00:00
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62714 sha256=cf7bcb3d7c01a95e03dc414e1a736946a4da5e4059b1acbe9a39fa9a8c16ca1e
  Stored in directory: /root/.cache/pip/wheels/6a/41/e1/4d212ede7c9752d8f7050d78fa3295304bc22b8fb530d9ebe4
Successfully built agents


In [27]:
!pip install agentic_patterns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 0.30.0
    Uninstalling groq-0.30.0:
      Successfully uninstalled groq-0.30.0


In [28]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content
MODEL = "llama-3.3-70b-versatile"

In [29]:
# define the system prompt as a constant
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Miami?</question>
<thought>I need to get the current weather in Miami</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 75, "unit": "fahrenheit"}}</observation>

You then output:

<response>The current temperature in Miami is 75 degrees Fahrenheit</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

In [30]:
# defining the tools
@tool
def sum_two_elements(a: int, b: int) -> int:
    """
    Computes the sum of two integers.

    Args:
        a (int): The first integer to be summed.
        b (int): The second integer to be summed.

    Returns:
        int: The sum of `a` and `b`.
    """
    return a + b


@tool
def multiply_two_elements(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a (int): The first integer to multiply.
        b (int): The second integer to multiply.

    Returns:
        int: The product of `a` and `b`.
    """
    return a * b

@tool
def compute_log(x: int) -> float | str:
    """
    Computes the logarithm of an integer `x` with an optional base.

    Args:
        x (int): The integer value for which the logarithm is computed. Must be greater than 0.

    Returns:
        float: The logarithm of `x` to the specified `base`.
    """
    if x <= 0:
        return "Logarithm is undefined for values less than or equal to 0."
    
    return math.log(x)


available_tools = {
    "sum_two_elements": sum_two_elements,
    "multiply_two_elements": multiply_two_elements,
    "compute_log": compute_log
}

In [31]:
# check the tool operator
print("Tool name: ", sum_two_elements.name)
print("Tool signature: ", sum_two_elements.fn_signature)

Tool name:  sum_two_elements
Tool signature:  {"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}}


In [32]:
# adding the tool signature to the system prompt
tools_signature = sum_two_elements.fn_signature + ",\n" + multiply_two_elements.fn_signature + ",\n" + compute_log.fn_signature

In [33]:
print(tools_signature)

{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "multiply_two_elements", "description": "\n    Multiplies two integers.\n\n    Args:\n        a (int): The first integer to multiply.\n        b (int): The second integer to multiply.\n\n    Returns:\n        int: The product of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "compute_log", "description": "\n    Computes the logarithm of an integer `x` with an optional base.\n\n    Args:\n        x (int): The integer value for which the logarithm is computed. Must be greater than 0.\n\n    Returns:\n        float: The logarithm of `x` to the specified `base`.\n    ", "parameters": {"prop

In [34]:
REACT_SYSTEM_PROMPT = REACT_SYSTEM_PROMPT % tools_signature

In [35]:
print(REACT_SYSTEM_PROMPT)


You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"p

In [36]:
# ReAct Loop step 1
USER_QUESTION = "I want to calculate the sum of 5897 and 9521 and multiply the result by 3. Then, I want to take the logarithm of this result"
chat_history = [
    {
        "role": "system",
        "content": REACT_SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": f"<question>{USER_QUESTION}</question>"
    }
]

In [37]:
output = GROQ_CLIENT.chat.completions.create(
    messages=chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<thought>I need to calculate the sum of 5897 and 9521, then multiply the result by 3, and finally compute the logarithm of this result.</thought>
<tool_call>{"name": "sum_two_elements","arguments": {"a": 5897, "b": 9521}, "id": 0}</tool_call>


In [38]:
chat_history.append(
    {
        "role": "assistant",
        "content": output
    }
)

In [39]:
# ReAct loop step 2
tool_call = extract_tag_content(output, tag="tool_call")

In [40]:
tool_call

TagContentResult(content=['{"name": "sum_two_elements","arguments": {"a": 5897, "b": 9521}, "id": 0}'], found=True)

In [41]:
tool_call = json.loads(tool_call.content[0])

In [42]:
tool_call

{'name': 'sum_two_elements', 'arguments': {'a': 5897, 'b': 9521}, 'id': 0}

In [43]:
tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])

In [44]:
assert tool_result == 5897 + 9521

In [45]:
chat_history.append(
    {
        "role": "user",
        "content": f"<observation>{tool_result}</observation>"
    }
)

In [46]:
# ReAct loop step 3
output = GROQ_CLIENT.chat.completions.create(
    messages=chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<thought>I have the sum of 5897 and 9521, which is 15418. Now, I need to multiply this result by 3.</thought>
<tool_call>{"name": "multiply_two_elements","arguments": {"a": 15418, "b": 3}, "id": 1}</tool_call>


In [47]:
chat_history.append(
    {
        "role": "assistant",
        "content": output
    }
)

In [48]:
# ReAct loop step 4
tool_call = extract_tag_content(output, tag="tool_call")
tool_call = json.loads(tool_call.content[0])
tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])

In [49]:
tool_result

46254

In [50]:
assert tool_result == (5897 + 9521) * 3

In [51]:
chat_history.append(
    {
        "role": "user",
        "content": f"<observation>{tool_result}</observation>"
    }
)

In [52]:
# ReAct loop step 5
output = GROQ_CLIENT.chat.completions.create(
    messages=chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<thought>I have the result of multiplying 15418 by 3, which is 46254. Now, I need to compute the logarithm of this result.</thought>
<tool_call>{"name": "compute_log","arguments": {"x": 46254}, "id": 2}</tool_call>


In [53]:
chat_history.append(
    {
        "role": "assistant",
        "content": output
    }
)

In [54]:
# ReAct loop step 6
tool_call = extract_tag_content(output, tag="tool_call")
tool_call = json.loads(tool_call.content[0])
tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])

In [55]:
assert tool_result == math.log((5897 + 9521) * 3)

In [56]:
chat_history.append(
    {
        "role": "user",
        "content": f"<observation>{tool_result}</observation>"
    }
)

In [57]:
# ReAct loop step 7
output = GROQ_CLIENT.chat.completions.create(
    messages=chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<thought>I have the logarithm of 46254, which is approximately 10.74. I can now provide the final answer.</thought>
<response>The sum of 5897 and 9521 is 15418. Multiplying this result by 3 gives 46254. The logarithm of 46254 is approximately 10.74.</response>


**Multi-Agent pattern**

In [58]:
from textwrap import dedent

from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew
from agentic_patterns.planning_pattern.react_agent import ReactAgent
from agentic_patterns.tool_pattern.tool import Tool


class Agent:
    """
    Represents an AI agent that can work as part of a team to complete tasks.

    This class implements an agent with dependencies, context handling, and task execution capabilities.
    It can be used in a multi-agent system where agents collaborate to solve complex problems.

    Attributes:
        name (str): The name of the agent.
        backstory (str): The backstory or background of the agent.
        task_description (str): A description of the task assigned to the agent.
        task_expected_output (str): The expected format or content of the task output.
        react_agent (ReactAgent): An instance of ReactAgent used for generating responses.
        dependencies (list[Agent]): A list of Agent instances that this agent depends on.
        dependents (list[Agent]): A list of Agent instances that depend on this agent.
        context (str): Accumulated context information from other agents.

    Args:
        name (str): The name of the agent.
        backstory (str): The backstory or background of the agent.
        task_description (str): A description of the task assigned to the agent.
        task_expected_output (str, optional): The expected format or content of the task output. Defaults to "".
        tools (list[Tool] | None, optional): A list of Tool instances available to the agent. Defaults to None.
        llm (str, optional): The name of the language model to use. Defaults to "llama-3.3-70b-versatile".
    """

    def __init__(
        self,
        name: Groq(api_key=secret_value_0),
        backstory: str,
        task_description: str,
        task_expected_output: str = "",
        tools: list[Tool] | None = None,
        llm: str = "llama-3.3-70b-versatile",
    ):
        self.name = name
        self.backstory = backstory
        self.task_description = task_description
        self.task_expected_output = task_expected_output
        self.react_agent = ReactAgent(
            model=llm, system_prompt=self.backstory, tools=tools or []
        )

        self.dependencies: list[Agent] = []  # Agents that this agent depends on
        self.dependents: list[Agent] = []  # Agents that depend on this agent

        self.context = ""

        # Automatically register this agent to the active Crew context if one exists
        Crew.register_agent(self)

    def __repr__(self):
        return f"{self.name}"

    def __rshift__(self, other):
        """
        Defines the '>>' operator. This operator is used to indicate agent dependency.

        Args:
            other (Agent): The agent that depends on this agent.
        """
        self.add_dependent(other)
        return other  # Allow chaining

    def __lshift__(self, other):
        """
        Defines the '<<' operator to indicate agent dependency in reverse.

        Args:
            other (Agent): The agent that this agent depends on.

        Returns:
            Agent: The `other` agent to allow for chaining.
        """
        self.add_dependency(other)
        return other  # Allow chaining

    def __rrshift__(self, other):
        """
        Defines the '<<' operator.This operator is used to indicate agent dependency.

        Args:
            other (Agent): The agent that this agent depends on.
        """
        self.add_dependency(other)
        return self  # Allow chaining

    def __rlshift__(self, other):
        """
        Defines the '<<' operator when evaluated from right to left.
        This operator is used to indicate agent dependency in the normal order.

        Args:
            other (Agent): The agent that depends on this agent.

        Returns:
            Agent: The current agent (self) to allow for chaining.
        """
        self.add_dependent(other)
        return self  # Allow chaining

    def add_dependency(self, other):
        """
        Adds a dependency to this agent.

        Args:
            other (Agent | list[Agent]): The agent(s) that this agent depends on.

        Raises:
            TypeError: If the dependency is not an Agent or a list of Agents.
        """
        if isinstance(other, Agent):
            self.dependencies.append(other)
            other.dependents.append(self)
        elif isinstance(other, list) and all(isinstance(item, Agent) for item in other):
            for item in other:
                self.dependencies.append(item)
                item.dependents.append(self)
        else:
            raise TypeError("The dependency must be an instance or list of Agent.")

    def add_dependent(self, other):
        """
        Adds a dependent to this agent.

        Args:
            other (Agent | list[Agent]): The agent(s) that depend on this agent.

        Raises:
            TypeError: If the dependent is not an Agent or a list of Agents.
        """
        if isinstance(other, Agent):
            other.dependencies.append(self)
            self.dependents.append(other)
        elif isinstance(other, list) and all(isinstance(item, Agent) for item in other):
            for item in other:
                item.dependencies.append(self)
                self.dependents.append(item)
        else:
            raise TypeError("The dependent must be an instance or list of Agent.")

    def receive_context(self, input_data):
        """
        Receives and stores context information from other agents.

        Args:
            input_data (str): The context information to be added.
        """
        self.context += f"{self.name} received context: \n{input_data}"

    def create_prompt(self):
        """
        Creates a prompt for the agent based on its task description, expected output, and context.

        Returns:
            str: The formatted prompt string.
        """
        prompt = dedent(
            f"""
        You are an AI agent. You are part of a team of agents working together to complete a task.
        I'm going to give you the task description enclosed in <task_description></task_description> tags. I'll also give
        you the available context from the other agents in <context></context> tags. If the context
        is not available, the <context></context> tags will be empty. You'll also receive the task
        expected output enclosed in <task_expected_output></task_expected_output> tags. With all this information
        you need to create the best possible response, always respecting the format as describe in
        <task_expected_output></task_expected_output> tags. If expected output is not available, just create
        a meaningful response to complete the task.

        <task_description>
        {self.task_description}
        </task_description>

        <task_expected_output>
        {self.task_expected_output}
        </task_expected_output>

        <context>
        {self.context}
        </context>

        Your response:
        """
        ).strip()

        return prompt

    def run(self):
        """
        Runs the agent's task and generates the output.

        This method creates a prompt, runs it through the ReactAgent, and passes the output to all dependent agents.

        Returns:
            str: The output generated by the agent.
        """
        msg = self.create_prompt()
        output = self.react_agent.run(user_msg=msg)

        # Pass the output to all dependents
        for dependent in self.dependents:
            dependent.receive_context(output)
        return output

In [59]:
import json
import re

from colorama import Fore
from dotenv import load_dotenv
from groq import Groq

from agentic_patterns.tool_pattern.tool import Tool
from agentic_patterns.tool_pattern.tool import validate_arguments
from agentic_patterns.utils.completions import build_prompt_structure
from agentic_patterns.utils.completions import ChatHistory
from agentic_patterns.utils.completions import completions_create
from agentic_patterns.utils.completions import update_chat_history
from agentic_patterns.utils.extraction import extract_tag_content

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")


load_dotenv()

BASE_SYSTEM_PROMPT = ""


REACT_SYSTEM_PROMPT = """
You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools>
%s
</tools>

Example session:

<question>What's the current temperature in Madrid?</question>
<thought>I need to get the current weather in Madrid</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>

You then output:

<response>The current temperature in Madrid is 25 degrees Celsius</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""


class ReactAgent:
    """
    A class that represents an agent using the ReAct logic that interacts with tools to process
    user inputs, make decisions, and execute tool calls. The agent can run interactive sessions,
    collect tool signatures, and process multiple tool calls in a given round of interaction.

    Attributes:
        client (Groq): The Groq client used to handle model-based completions.
        model (str): The name of the model used for generating responses. Default is "llama-3.3-70b-versatile".
        tools (list[Tool]): A list of Tool instances available for execution.
        tools_dict (dict): A dictionary mapping tool names to their corresponding Tool instances.
    """

    def __init__(
        self,
        tools: Tool | list[Tool],
        model: str = "llama-3.3-70b-versatile",
        system_prompt: str = BASE_SYSTEM_PROMPT,
    ) -> None:
        self.client = Groq(api_key=secret_value_0)
        self.model = model
        self.system_prompt = system_prompt
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {tool.name: tool for tool in self.tools}

    def add_tool_signatures(self) -> str:
        """
        Collects the function signatures of all available tools.

        Returns:
            str: A concatenated string of all tool function signatures in JSON format.
        """
        return "".join([tool.fn_signature for tool in self.tools])

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        """
        Processes each tool call, validates arguments, executes the tools, and collects results.

        Args:
            tool_calls_content (list): List of strings, each representing a tool call in JSON format.

        Returns:
            dict: A dictionary where the keys are tool call IDs and values are the results from the tools.
        """
        observations = {}
        for tool_call_str in tool_calls_content:
            tool_call = json.loads(tool_call_str)
            tool_name = tool_call["name"]
            tool = self.tools_dict[tool_name]

            print(Fore.GREEN + f"\nUsing Tool: {tool_name}")

            # Validate and execute the tool call
            validated_tool_call = validate_arguments(
                tool_call, json.loads(tool.fn_signature)
            )
            print(Fore.GREEN + f"\nTool call dict: \n{validated_tool_call}")

            result = tool.run(**validated_tool_call["arguments"])
            print(Fore.GREEN + f"\nTool result: \n{result}")

            # Store the result using the tool call ID
            observations[validated_tool_call["id"]] = result

        return observations

    def run(
        self,
        user_msg: str,
        max_rounds: int = 10,
    ) -> str:
        """
        Executes a user interaction session, where the agent processes user input, generates responses,
        handles tool calls, and updates chat history until a final response is ready or the maximum
        number of rounds is reached.

        Args:
            user_msg (str): The user's input message to start the interaction.
            max_rounds (int, optional): Maximum number of interaction rounds the agent should perform. Default is 10.

        Returns:
            str: The final response generated by the agent after processing user input and any tool calls.
        """
        user_prompt = build_prompt_structure(
            prompt=user_msg, role="user", tag="question"
        )
        if self.tools:
            self.system_prompt += (
                "\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures()
            )

        chat_history = ChatHistory(
            [
                build_prompt_structure(
                    prompt=self.system_prompt,
                    role="system",
                ),
                user_prompt,
            ]
        )

        if self.tools:
            # Run the ReAct loop for max_rounds
            for _ in range(max_rounds):

                completion = completions_create(self.client, chat_history, self.model)

                response = extract_tag_content(str(completion), "response")
                if response.found:
                    return response.content[0]

                thought = extract_tag_content(str(completion), "thought")
                tool_calls = extract_tag_content(str(completion), "tool_call")

                update_chat_history(chat_history, completion, "assistant")

                print(Fore.MAGENTA + f"\nThought: {thought.content[0]}")

                if tool_calls.found:
                    observations = self.process_tool_calls(tool_calls.content)
                    print(Fore.BLUE + f"\nObservations: {observations}")
                    update_chat_history(chat_history, f"{observations}", "user")

        return completions_create(self.client, chat_history, self.model)